## **Loading Data from News table from the Lakehouse DB**

In [1]:
df = spark.sql("SELECT * FROM bing_lake_db.tbl_latest_news")
display(df)

StatementMeta(, 480d8b6a-e513-4c52-826e-968c1644a3fe, 3, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, dfee60d7-cff5-4c86-b114-2480d1772ad2)

#### **Importing Synapse ML**

In [2]:
import synapse.ml.core
from synapse.ml.services import AnalyzeText

StatementMeta(, 480d8b6a-e513-4c52-826e-968c1644a3fe, 4, Finished, Available, Finished)

#### **Configuring Synapse ML**

In [3]:
# Importing the model and configuring the cols for our use:

model = (AnalyzeText()
        .setTextCol("descrption")
        .setKind("SentimentAnalysis")
        .setOutputCol("response")
        .setErrorCol("error"))

StatementMeta(, 480d8b6a-e513-4c52-826e-968c1644a3fe, 5, Finished, Available, Finished)

#### **Running Sentiment Analysis on our data**

In [4]:
# Applying the model to our dataframe
result = model.transform(df)

StatementMeta(, 480d8b6a-e513-4c52-826e-968c1644a3fe, 6, Finished, Available, Finished)

In [5]:
display(result)

StatementMeta(, 480d8b6a-e513-4c52-826e-968c1644a3fe, 7, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, d22835ce-8820-4f72-97ec-c8d2bdb883e0)

#### **Adding Sentiment Column from the result col and removing the unwanted columns**

In [6]:
#New Sentiment Column
from pyspark.sql.functions import col
sentiment_df = result.withColumn("sentiment", col("response.documents.sentiment"))

StatementMeta(, 480d8b6a-e513-4c52-826e-968c1644a3fe, 8, Finished, Available, Finished)

In [7]:
display(sentiment_df)

StatementMeta(, 480d8b6a-e513-4c52-826e-968c1644a3fe, 9, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 5a67a746-152e-491b-8c88-0d1b237bec53)

In [8]:
sent_df_final = sentiment_df.drop("error","response")

StatementMeta(, 480d8b6a-e513-4c52-826e-968c1644a3fe, 10, Finished, Available, Finished)

In [9]:
display(sent_df_final)

StatementMeta(, 480d8b6a-e513-4c52-826e-968c1644a3fe, 11, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, f1afef36-1fac-400c-a0fe-2ebd1b0ca489)

#### **Creating New File with the Sentiment Analysis completed back into the DB Using Type1 Merge**

In [10]:
from pyspark.sql.utils import AnalysisException

try:
    table_name = 'bing_lake_db.tbl_sentiment_analysis'
    sent_df_final.write.format("delta").saveAsTable(table_name)

except AnalysisException:
    print("Table Already Exists")

    sent_df_final.createOrReplaceTempView("vw_sent_df_final")

    spark.sql(f""" MERGE INTO {table_name} target_table
                    USING vw_sent_df_final source_view

                    ON source_view.url = target_table.url

                    WHEN MATCHED AND
                    source_view.title <> target_table.title OR
                    source_view.descrption <> target_table.descrption OR
                    source_view.category <> target_table.category OR
                    source_view.image <> target_table.image OR
                    source_view.provider <> target_table.provider OR
                    source_view.datePublished <> target_table.datePublished 
                    
                    THEN UPDATE SET *
                    WHEN NOT MATCHED THEN INSERT *
                """)

StatementMeta(, 480d8b6a-e513-4c52-826e-968c1644a3fe, 12, Finished, Available, Finished)